In [152]:
import mlflow
import pandas as pd
import os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [153]:
db_path = os.path.abspath("mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
mlflow.set_experiment("nba_betting_models_test")

2026/05/29 21:50:31 INFO mlflow.tracking.fluent: Experiment with name 'nba_betting_models_test' does not exist. Creating a new experiment.


Tracking URI: sqlite:////Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/notebooks/mlflow.db


<Experiment: artifact_location=('/Users/abhaybapat/Desktop/Data Science '
 'Project/betting_classification_model/notebooks/mlruns/2'), creation_time=1780109431955, experiment_id='2', last_update_time=1780109431955, lifecycle_stage='active', name='nba_betting_models_test', tags={}, trace_location=None, workspace='default'>

In [154]:
df = pd.read_csv("../data/final_dataset.csv", index_col=0)

In [155]:
df["game_date"] = pd.DataFrame(df["game_date"])
df = df.sort_values(by="game_date")

In [156]:
df.head()

,game_date,pre_game_elo_home,is_B2B_home,pre_game_elo_away,is_B2B_away,pre_game_elo_diff,days_rest_diff,possessions_rolling_diff,eFG_rolling_diff,TO%_rolling_diff,OREB%_rolling_diff,FTR_rolling_diff,off_rating_rolling_diff,def_rating_rolling_diff,net_rating_rolling_diff,win_home
0,1986-11-01 20:00:00,1421.87,True,1448.97,True,-27.10,0.0,-10.9824,-0.099275,-0.086457,-0.091463,-0.002415,-8.464992,-3.910563,-4.554429,True
1,1986-11-01 20:00:00,1431.30,True,1518.94,True,-87.64,0.0,-8.0640,0.094880,-0.027357,-0.040309,-0.089677,8.659146,-2.711930,11.371076,True
2,1986-11-01 20:00:00,1471.80,True,1463.65,True,8.15,0.0,0.9984,0.161866,0.007656,-0.051724,-0.095930,18.519900,2.888894,15.631006,True
3,1986-11-01 20:00:00,1443.05,True,1524.34,True,-81.29,0.0,4.9920,-0.044118,0.040420,-0.180180,0.058229,-16.170137,16.614965,-32.785102,True
4,1986-11-01 20:00:00,1496.85,True,1461.72,True,35.13,0.0,4.7232,-0.195382,0.000538,-0.018490,-0.147186,-36.429595,-28.609532,-7.820063,True


In [157]:
X = df.drop(columns=["game_date", "win_home"])
baseline_X = df[["pre_game_elo_diff"]]
y = df["win_home"]

In [158]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=42)
X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline = train_test_split(baseline_X, y, test_size=0.2, shuffle=False, random_state=42)

In [159]:
baseline_pipeline = Pipeline([
    ("scaler", StandardScaler()), 
    ("lr", LogisticRegression())
])

with mlflow.start_run(run_name="baseline_model"):
    baseline_pipeline.fit(X_train_baseline, y_train_baseline)
    
    y_pred_baseline = baseline_pipeline.predict(X_test_baseline)
    y_prob_baseline = baseline_pipeline.predict_proba(X_test_baseline)[:, 1]
    
    mlflow.log_metric("accuracy", accuracy_score(y_test_baseline, y_pred_baseline))
    mlflow.log_metric("log_loss", log_loss(y_test_baseline, y_prob_baseline))
    mlflow.log_metric("brier_score_loss", brier_score_loss(y_test_baseline, y_prob_baseline))
    
    mlflow.sklearn.log_model(sk_model=baseline_pipeline, name="baseline_model")



2026/05/29 21:50:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [160]:
models = {
    "RandomForest": Pipeline([("scaler", StandardScaler()), ("clf", RandomForestClassifier(criterion="log_loss"))]),
    "XGBoost": Pipeline([("scaler", StandardScaler()), ("clf", XGBClassifier(objective="binary:logistic", eval="logloss"))]),
    "LGBM": Pipeline([("scaler", StandardScaler()), ("clf", LGBMClassifier(objective="binary", metric="binary_logloss"))]),
    "LogisticRegression": Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression())])
}

In [161]:
for model_name, pipeline in models.items():
    with mlflow.start_run(run_name=model_name):
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        
        mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
        mlflow.log_metric("log_loss", log_loss(y_test, y_prob))
        mlflow.log_metric("brier_score_loss", brier_score_loss(y_test, y_prob))
        
        mlflow.sklearn.log_model(sk_model=pipeline, name=model_name)
        
        mlflow.log_param("model_name", model_name)

2026/05/29 21:50:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[21:50:48] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "eval" } are not used.

2026/05/29 21:50:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[LightGBM] [Info] Number of positive: 25052, number of negative: 16000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000355 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2921
[LightGBM] [Info] Number of data points in the train set: 41052, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.610250 -> initscore=0.448365
[LightGBM] [Info] Start training from score 0.448365


/Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/05/29 21:50:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/29 21:50:52 WARNING mlflow.sklearn: Saving scikit-learn models in the

In [162]:
# mlflow ui --backend-store-uri "sqlite:////Users/abhaybapat/Desktop/Data Science Project/betting_classification_model/notebooks/mlflow.db" --port 5001